# Day 2 — The Four State Points of a Refrigeration Cycle
**Mahesha Gonal — Refrigeration Simulation Portfolio**

**In plain terms:** every vapour-compression refrigerator runs the refrigerant through four stops, in a loop:

1. **Point 1** — cold low-pressure vapour leaving the evaporator (about to enter the compressor)
2. **Point 2** — hot high-pressure vapour leaving the compressor (about to enter the condenser)
3. **Point 3** — warm high-pressure liquid leaving the condenser (about to enter the capillary tube)
4. **Point 4** — cold low-pressure liquid/vapour mix entering the evaporator

This notebook calculates the energy (enthalpy, `h`) at each of these four points for **R134a**, running between a **-20°C evaporator** (a freezer-type duty) and a **40°C condenser** (a typical Indian ambient + heat-rejection margin), then uses those four numbers to get the cycle's efficiency (**COP**).

In [1]:
# CELL 1 — Install dependencies (run this first, every time)
!pip install coolprop matplotlib numpy PyGithub -q

In [2]:
# CELL 2 — Imports
import CoolProp.CoolProp as CP

In [3]:
# CELL 3 — Four state points: R134a | T_evap=-20C | T_cond=40C
refrigerant = 'R134a'
T_evap = -20 + 273.15
T_cond = 40 + 273.15

P_evap = CP.PropsSI('P', 'T', T_evap, 'Q', 1, refrigerant)
P_cond = CP.PropsSI('P', 'T', T_cond, 'Q', 1, refrigerant)

h1 = CP.PropsSI('H', 'P', P_evap, 'Q', 1, refrigerant) / 1000   # evaporator exit (sat. vapour)
s1 = CP.PropsSI('S', 'P', P_evap, 'Q', 1, refrigerant)
h2 = CP.PropsSI('H', 'P', P_cond, 'S', s1, refrigerant) / 1000  # compressor exit (isentropic)
h3 = CP.PropsSI('H', 'P', P_cond, 'Q', 0, refrigerant) / 1000   # condenser exit (sat. liquid)
h4 = h3                                                          # after capillary tube (h unchanged)

print(f"P_evap = {P_evap/1e5:.2f} bar   P_cond = {P_cond/1e5:.2f} bar")
print(f"h1 = {h1:.2f}   h2 = {h2:.2f}   h3 = {h3:.2f}   h4 = {h4:.2f}  (all kJ/kg)")

P_evap = 1.33 bar   P_cond = 10.17 bar
h1 = 386.55   h2 = 429.04   h3 = 256.41   h4 = 256.41  (all kJ/kg)


In [4]:
# CELL 4 — Energy balance -> COP
Q_evap = h1 - h4   # refrigerating effect (cooling delivered per kg of refrigerant)
W_comp = h2 - h1   # compressor work (electrical-to-mechanical work needed, per kg)
COP = Q_evap / W_comp

print(f"Refrigerating effect: {Q_evap:.2f} kJ/kg")
print(f"Compressor work:      {W_comp:.2f} kJ/kg")
print(f"COP:                  {COP:.3f}")

Refrigerating effect: 130.15 kJ/kg
Compressor work:      42.48 kJ/kg
COP:                  3.064


**What this number means:** a COP of around 2 means that for every 1 unit of electrical work the compressor uses, the cycle pulls about 2 units of heat out of the cabin. Higher COP = more cooling per rupee of electricity — this single number is the core efficiency metric that BEE star ratings are ultimately built from.

Day 4 will take these same four points and draw them on a pressure–enthalpy map, so the cycle becomes a picture instead of just numbers.

In [ ]:
# FINAL CELL — Push this notebook to GitHub
from github import Github, Auth
from google.colab import userdata, _message
import json

token = userdata.get('GITHUB_TOKEN')
auth = Auth.Token(token)
g = Github(auth=auth)
repo = g.get_repo("MaheshaGonal/refrigeration-simulation-python")

nb_data = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
content = json.dumps(nb_data['ipynb'], indent=1)

filename = "day02_state_points.ipynb"

try:
    existing = repo.get_contents(filename)
    repo.update_file(filename, "Add Day 02 - four cycle state points + COP", content, existing.sha)
    print("Updated existing file on GitHub")
except Exception:
    repo.create_file(filename, "Add Day 02 - four cycle state points + COP", content)
    print("Created new file on GitHub")

print("GitHub repo: https://github.com/MaheshaGonal/refrigeration-simulation-python")
